In [ ]:
# ==========================================
# 0. INSTALLATION DES DÉPENDANCES
# ==========================================
print("Installation des bibliothèques nécessaires...")
!pip install -q pystac-client planetary-computer stackstac xarray rioxarray pandas numpy matplotlib

# ==========================================
# 1. IMPORTATION DES MODULES
# ==========================================
import os
import planetary_computer
from pystac_client import Client
import stackstac
import xarray as xr
import pandas as pd
import numpy as np
from google.colab import drive

print("Modules importés avec succès !")

# ==========================================
# 2. CONNEXION AU GOOGLE DRIVE
# ==========================================
print("Connexion à Google Drive...")
drive.mount('/content/drive')

# Définition du dossier de sauvegarde sur votre Drive
output_dir = '/content/drive/MyDrive/TerraPulse_Data'
os.makedirs(output_dir, exist_ok=True)
output_csv_path = os.path.join(output_dir, 'bukavu_terrapulse_data.csv')

# ==========================================
# 3. DÉFINITION DES PARAMÈTRES DE LA ZONE (BBOX) & TEMPS
# ==========================================
# Zone géographique : Région de Bukavu / Lac Kivu [min_lon, min_lat, max_lon, max_lat]
bukavu_bbox = [28.8, -2.6, 29.2, -2.2]

# Période d'étude (ex: Janvier 2021 à Décembre 2025)
time_range = "2021-01-01/2025-12-31"

print(f"Zone cible : Bukavu (BBOX: {bukavu_bbox})")
print(f"Période d'étude : {time_range}")

# ==========================================
# 4. REQUÊTE AU CATALOGUE STAC (MICROSOFT PLANETARY COMPUTER)
# ==========================================
print("Connexion au catalogue STAC de Microsoft Planetary Computer...")
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Recherche dans la collection Sentinel-2 L2A (pour un suivi fin de la surface / végétation)
print("Recherche des images satellites disponibles...")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bukavu_bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 30}} # Filtrer les images avec moins de 30% de nuages
)

items = search.item_collection()
print(f"Nombre d'images trouvées : {len(items)}")

# ==========================================
# 5. EXTRACTION ET TRAITEMENT DES DONNÉES
# ==========================================
if len(items) > 0:
    print("Extraction et chargement des matrices de données spatiales...")

    # Chargement d'un échantillon (ou des 10 premières images pour limiter la mémoire du notebook)
    # Bandes : B04 (Rouge), B08 (Proche Infrarouge pour le NDVI)
    stack = stackstac.stack(
        items[:10],
        bounds_latlon=bukavu_bbox,
        epsg=4326,
        resolution=0.001, # Résolution grossière pour aller vite, ex: ~100m
        assets=["B04", "B08"]
    )

    # Calcul d'une série temporelle moyenne sur la zone (Indice NDVI et valeurs brutes)
    # NDVI = (NIR - Red) / (NIR + Red) -> (B08 - B04) / (B08 + B04)
    red = stack.sel(band="B04")
    nir = stack.sel(band="B08")
    ndvi = (nir - red) / (nir + red)

    # Moyenne spatiale par date pour obtenir une série temporelle tabulaire
    ndvi_mean_time = ndvi.mean(dim=["y", "x"]).compute()

    # Création du DataFrame Pandas
    df_result = pd.DataFrame({
        'date': pd.to_datetime(ndvi_mean_time.time.values),
        'ndvi_ moyen': ndvi_mean_time.values,
        'region': 'Bukavu_Kivu'
    })

    # Nettoyage des valeurs nulles/NaN éventuelles
    df_result = df_result.dropna()

else:
    print("Aucune image trouvée avec ces critères stricts. Génération d'un jeu de données de secours basé sur les tendances historiques simulées pour le prototype.")
    # Fallback pour s'assurer que le fichier CSV est toujours généré pour le hackathon
    dates = pd.date_range(start="2021-01-01", end="2025-12-31", freq="M")
    df_result = pd.DataFrame({
        'date': dates,
        'precipitation_gpm_est': np.random.uniform(50, 250, len(dates)),
        'humidite_sol_smap': np.random.uniform(20, 80, len(dates)),
        'ndvi_moyen': np.random.uniform(0.4, 0.8, len(dates)),
        'region': 'Bukavu_Kivu'
    })

# ==========================================
# 6. EXPORTATION VERS GOOGLE DRIVE EN CSV
# ==========================================
df_result.to_csv(output_csv_path, index=False)

print("\n" + "="*50)
print(f"SUCCÈS ! Fichier CSV généré et sauvegardé avec succès.")
print(f"Emplacement dans votre Drive : {output_csv_path}")
print("="*50)

# Affichage des 5 premières lignes du tableau obtenu
display(df_result.head())

Installation des bibliothèques nécessaires...
Modules importés avec succès !
Connexion à Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zone cible : Bukavu (BBOX: [28.8, -2.6, 29.2, -2.2])
Période d'étude : 2021-01-01/2025-12-31
Connexion au catalogue STAC de Microsoft Planetary Computer...
Recherche des images satellites disponibles...
Nombre d'images trouvées : 193
Extraction et chargement des matrices de données spatiales...



SUCCÈS ! Fichier CSV généré et sauvegardé avec succès.
Emplacement dans votre Drive : /content/drive/MyDrive/TerraPulse_Data/bukavu_terrapulse_data.csv


,date,ndvi_ moyen,region
0,2025-08-14 08:06:31.025,0.208515,Bukavu_Kivu
1,2025-08-19 08:06:09.024,0.209607,Bukavu_Kivu
2,2025-08-19 08:06:09.024,0.228118,Bukavu_Kivu
3,2025-08-24 08:06:31.025,0.185855,Bukavu_Kivu
4,2025-08-29 08:06:09.024,0.219749,Bukavu_Kivu


In [ ]:
import pandas as pd
import numpy as np
import os

print("Génération d'un jeu de données complet et structuré pour Bukavu (2021 - 2025)...")

# 1. Création d'une plage temporelle complète (mensuelle) de Janvier 2021 à Décembre 2025
dates = pd.date_range(start="2021-01-01", end="2025-12-31", freq="ME")

# 2. Simulation de données physiques cohérentes avec le climat de Bukavu (climat tropical de montagne)
# Bukavu connaît une saison des pluies (importante de septembre à mai) et une saison sèche (juin à août)
np.random.seed(42)

precipitations = []
humidite_sol = []
ndvi_values = []

for date in dates:
    mois = date.month
    # Simulation de la saison sèche (juin, juillet, août)
    if mois in [6, 7, 8]:
        precip = np.random.uniform(15, 50)          # Faibles pluies en mm
        humid = np.random.uniform(25, 45)           # Humidité du sol basse en %
        ndvi = np.random.uniform(0.55, 0.68)        # Végétation légèrement moins active
    # Saison des pluies (le reste de l'année)
    else:
        precip = np.random.uniform(100, 220)        # Fortes précipitations en mm
        humid = np.random.uniform(55, 85)           # Humidité du sol élevée en %
        ndvi = np.random.uniform(0.68, 0.82)        # Végétation très active

    precipitations.append(round(precip, 1))
    humidite_sol.append(round(humid, 1))
    ndvi_values.append(round(ndvi, 3))

# 3. Construction du DataFrame avec des colonnes claires et explicites
df_complet = pd.DataFrame({
    'date': dates.strftime('%Y-%m-%d'),
    'precipitation_mensuelle_mm': precipitations,
    'humidite_sol_pourcent': humidite_sol,
    'ndvi_moyen_vegetation': ndvi_values,
    'region': 'Bukavu_Kivu'
})

# 4. Sauvegarde du fichier CSV complet sur Google Drive
output_csv_path_complet = '/content/drive/MyDrive/TerraPulse_Data/bukavu_terrapulse_complet_2021_2025.csv'
os.makedirs(os.path.dirname(output_csv_path_complet), exist_ok=True)
df_complet.to_csv(output_csv_path_complet, index=False)

print("\n" + "="*50)
print("SUCCÈS : Fichier CSV complet et structuré généré !")
print(f"Sauvegardé sous : {output_csv_path_complet}")
print(f"Nombre total de lignes (mois) : {len(df_complet)}")
print("="*50)

# Affichage des premières lignes pour vérification
display(df_complet.head(12))

Génération d'un jeu de données complet et structuré pour Bukavu (2021 - 2025)...

SUCCÈS : Fichier CSV complet et structuré généré !
Sauvegardé sous : /content/drive/MyDrive/TerraPulse_Data/bukavu_terrapulse_complet_2021_2025.csv
Nombre total de lignes (mois) : 60


,date,precipitation_mensuelle_mm,humidite_sol_pourcent,ndvi_moyen_vegetation,region
0,2021-01-31,144.9,83.5,0.782,Bukavu_Kivu
1,2021-02-28,171.8,59.7,0.702,Bukavu_Kivu
2,2021-03-31,107.0,81.0,0.764,Bukavu_Kivu
3,2021-04-30,185.0,55.6,0.816,Bukavu_Kivu
4,2021-05-31,199.9,61.4,0.705,Bukavu_Kivu
5,2021-06-30,21.4,31.1,0.618,Bukavu_Kivu
6,2021-07-31,30.1,30.8,0.630,Bukavu_Kivu
7,2021-08-31,19.9,30.8,0.598,Bukavu_Kivu
8,2021-09-30,154.7,78.6,0.708,Bukavu_Kivu
9,2021-10-31,161.7,72.8,0.687,Bukavu_Kivu
